# Model Comparison Notebook

This notebook will compare the baseline logistic regression model against a random forest classifier using training-data cross-validation. The original data prep and train/test split are reproduced to keep it consistent. The held-out test set is not used for model selection or threshold impact.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/creditsense_applications.csv")

print("Original dataset shape:", df.shape)

Importing the dataset and confirming it is the expected shape.

In [ ]:
df_model = df.dropna(
    subset=["savings_balance_gbp", "years_at_current_address"]
).copy()

print("Modelling dataset shape:", df_model.shape)
print("Rows excluded:", len(df) - len(df_model))

Removing the known missing data and confirming only those have been removed.

In [ ]:
excluded_features = [
    "application_id",
    "default_within_24m",
    "age_years",
    "sex",
    "marital_status",
    "residency_status",
    "num_dependents",
    "region",
    "loan_purpose",
    "application_channel",
    "previous_application_declined"
]

Duplicating the `excluded_features` list used in my baseline model.

In [ ]:
X = df_model.drop(columns=excluded_features)

y = df_model["default_within_24m"]

print("predictive features:", X.shape[1])
print("Target:", y.name)
print("\nSelected features:")
print(X.columns.tolist())

Duplicating the predictive features list and ensuring it has been successful.

In [ ]:
nominal_features = [
    "employment_status",
    "housing_status",
    "credit_history",
    "checking_account_status"
]

binary_features = [
    "has_savings_account",
    "has_guarantor"
]

numerical_features = [
    col for col in X.columns
    if col not in nominal_features + binary_features
]

Recreating the original feature split in order to define how I handle each.

In [ ]:
print("Nominal:", nominal_features)
print("Binary:", binary_features)
print("Numerical:", numerical_features)

print("\nTotal features:",
      len(nominal_features)
      + len(binary_features)
      + len(numerical_features))

Confirming the split has been successful.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Ran exactly the same split as in my baseline logistic regression model.

In [ ]:
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining default rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting default rate:")
print(y_test.value_counts(normalize=True))

Ensuring the split has been successful.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

Imported all 3 transformers ready for comparison.

In [ ]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

Took the preprocessing method from the baseline model and simply renamed and adjusted the numerical method. I chose to not use scaling on the numerical features in this model as it will respond better for the decision tree.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

Import the pipeline and model type.

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=42
        ))
    ]
)

Created the model pipeline but have not fitted it to any data yet. Also important to note that as random forests use a different method entirely we have used different training hyperparameters.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Recreated the cross-validation configuration from my baseline.

In [ ]:
rf_cv_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("ROC_AUC per fold:", rf_cv_scores)
print("Mean ROC_AUC:", rf_cv_scores.mean())
print("Standard deviation:", rf_cv_scores.std())

Evaluation of the random forest shows worse scores upon initial inspection compared with logistic regression. Also note we have not used `.fit()` yet as when using cross-validation it is doing it naturally in isolation within each fold.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

Exact copy of preprocessor cell from baseline model.

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline(
    steps =[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

Exact copy of the model cell in the baseline but added the import for `LogisticRegression`.

In [ ]:
cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("ROC_AUC per fold:", cv_scores)
print("Mean ROAC_AUC:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

Exact copy of the `cross_val_scores` section from the baseline model in order to get a comparison.

In [ ]:
from sklearn.model_selection import cross_val_predict

Importing the cross validation prediction method in order to determine tangible results for our stakeholder.

In [ ]:
lr_cv_predictions = cross_val_predict(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    method="predict"
)

Generating one class prediction for each applicant remaining using the cross validation fold data for logistic regression.

In [ ]:
rf_cv_predictions = cross_val_predict(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict"
)

Exactly the same but for random forests.

In [ ]:
from sklearn.metrics import confusion_matrix

lr_cm = confusion_matrix(y_train, lr_cv_predictions)
rf_cm = confusion_matrix(y_train, rf_cv_predictions)

print("Logistic Regression:")
print(lr_cm)

print("Random Forest:")
print(rf_cm)

Evidence that Logistic Regression has a stronger default recall at the default threshold in comparison to Random Forests.

## Threshold sensitivity analysis

In [ ]:
lr_cv_probabilities = cross_val_predict(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)

Generating Logistic Regression probabilities.

In [ ]:
lr_cv_probabilities[:5]

Viewing the first five of those probabilities.

In [ ]:
lr_default_probabilities = lr_cv_probabilities[:, 1]

Generating the probabilities to show just the default values.

In [ ]:
lr_default_probabilities[:5]

Viewing the first five of the default probabilities.

In [ ]:
rf_cv_probabilities = cross_val_predict(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)

Generating the Random Forest probabilities.

In [ ]:
rf_cv_probabilities[:5]

Viewing the first five of those.

In [ ]:
rf_default_probabilities = rf_cv_probabilities[:, 1]

Generating the probabilities for just the defaults in Random Forest model.

In [ ]:
rf_default_probabilities[:5]

Viewing just the first five of the default probabilities for Random Forest model. Note: Random Forest has given us results to only two decimal places i comparison to Logistic Regressions eight. This could impact our comparison.

In [ ]:
rf_pipeline.get_params()

Confirmed reasoning for less decimals is due to `n_estimators` being set to `100`. This is not an issue at this stage.

In [ ]:
lr_threshold_predictions = (lr_default_probabilities >= 0.5).astype(int)

Applied a threshold for comparison. Starting at 0.5 to confirm the defaults are as we expect them to be.

In [ ]:
lr_threshold_cm = confusion_matrix(y_train, lr_threshold_predictions)

print("Logistic Regression:")
print(lr_threshold_cm)

An exact match when comparing to our original.

In [ ]:
rf_threshold_predictions = (rf_default_probabilities >= 0.5).astype(int)

In [ ]:
rf_threshold_cm = confusion_matrix(y_train, rf_threshold_predictions)

print("Random Forest:")
print(rf_threshold_cm)

Ran the same process for Random Forest which logically should have produced the same result as the original however we have a net increase of 34 declines. This is likely due to the results of exactly 0.5 being moved to class 0 thanks to Scikit-learn's defaults for tie's.

In [ ]:
(rf_default_probabilities == 0.5).sum()

Confirmed there are 34 exact 0.5 results which causes the the method to auto shift the classification.

In [ ]:
rf_strict_predictions = (rf_default_probabilities > 0.5).astype(int)

confusion_matrix(y_train, rf_strict_predictions)

Confirmed our hypothesis above by using the `>` operator only.

In [ ]:
import numpy as np

thresholds = np.linspace(0.4, 0.6, 11).round(2)

print(thresholds)

Creating our thresholds values.

In [ ]:
threshold_results = []

Create the list for storing the results.

In [ ]:
from sklearn.metrics import f1_score, recall_score

for threshold in thresholds:
    predictions = (lr_default_probabilities >= threshold).astype(int)
    cm = confusion_matrix(y_train, predictions)
    tn, fp, fn, tp = cm.ravel()
    f1 = f1_score(y_train, predictions)
    recall = recall_score(y_train, predictions)
    threshold_results.append({
        "Model": "Logistic Regression",
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Recall": recall,
        "F1": f1
    })

Re-using the previous code but put the `thresholds` value in to automate the process. Included the extra scores and defined each into a dictionary.

In [ ]:
threshold_df = pd.DataFrame(threshold_results)
threshold_df

### Notes:

- Both key metrics (F1 and Recall) steadily decrease as we increase the threshold.
- No peak seen within this threshold range.
- At 0.4 the model is able to identify 161 more potential defaulters, but 208 more non-defaulters are incorrectly identified.
- At 0.4 we see a 14.05 percentage point improvement for recall and 5.47 percentage point improvement for F1 score.
- The potential impact from a business perspective of rejecting more potential safe applicants is significant.

In [ ]:
threshold_df["FPR"] = threshold_df["FP"] / (threshold_df["FP"] + threshold_df["TN"])

threshold_df

In [ ]:
threshold_df["FNR"] = threshold_df["FN"] / (threshold_df["FN"] + threshold_df["TP"])

threshold_df

In [ ]:
threshold_df["Precision"] = threshold_df["TP"] / (threshold_df["TP"] + threshold_df["FP"])

threshold_df

In [ ]:
threshold_df["Decline Rate"] = (threshold_df["FP"] + threshold_df["TP"]) / (threshold_df["FP"] + threshold_df["TP"] + threshold_df["TN"] + threshold_df["FN"])

threshold_df

In [ ]:
threshold_df["Cost 1:1"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 1)
threshold_df["Cost 1:2"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 2)
threshold_df["Cost 1:5"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 5)
threshold_df["Cost 1:10"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 10)

threshold_df

Please see decision journal entry 010 for further analysis of the above table.

In [ ]:
from sklearn.metrics import f1_score, recall_score

for threshold in thresholds:
    predictions = (rf_default_probabilities >= threshold).astype(int)
    cm = confusion_matrix(y_train, predictions)
    tn, fp, fn, tp = cm.ravel()
    f1 = f1_score(y_train, predictions)
    recall = recall_score(y_train, predictions)
    threshold_results.append({
        "Model": "Random Forest",
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Recall": recall,
        "F1": f1
    })

Regenerating the loop ready for random forest comparison.

In [ ]:
len(threshold_results)

Confirming I havent made any errors in the above.

In [ ]:
threshold_df = pd.DataFrame(threshold_results)
threshold_df

In [ ]:
threshold_df["FPR"] = threshold_df["FP"] / (threshold_df["FP"] + threshold_df["TN"])

threshold_df

In [ ]:
threshold_df["FNR"] = threshold_df["FN"] / (threshold_df["FN"] + threshold_df["TP"])

threshold_df

In [ ]:
threshold_df["Precision"] = threshold_df["TP"] / (threshold_df["TP"] + threshold_df["FP"])

threshold_df

In [ ]:
threshold_df["Decline Rate"] = (threshold_df["FP"] + threshold_df["TP"]) / (threshold_df["FP"] + threshold_df["TP"] + threshold_df["TN"] + threshold_df["FN"])

threshold_df

In [ ]:
threshold_df["Cost 1:1"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 1)
threshold_df["Cost 1:2"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 2)
threshold_df["Cost 1:5"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 5)
threshold_df["Cost 1:10"] = (threshold_df["FP"] * 1) + (threshold_df["FN"] * 10)

threshold_df

Recalculated all the additional metrics for a proper comparison.

## Notes:

- Logistic Regression produces 46 fewer classification errors overall when comaring each model at its lowest total error count.
- Random Forest incorrectly declines 19 less applicants at the lowest total error count.
- When using the ratio cost Logistic Regression acheives a lower minimum accross the board.
- This is not an exhaustive tuning of the Random Forest model or of the thresholds possible.

In [ ]:
threshold_df["Model"].value_counts()

Double checking no duplicates have been accidentally created.